# ASD-XMF queue orchestrator (runs on Kaggle CPU, no GPU quota)
Watches the account and pushes the ASD-XMF GPU notebooks one by one, **only after the
harness / any other notebook has finished** and the account has been quiet for 15 minutes.
Before Kaggle's 12-hour limit it relaunches itself, so the queue keeps going while your PC
and Claude are off.  It stops by itself when every notebook is done.

One-time setup: *Add-ons -> Secrets* -> add `KAGGLE_API_TOKEN` (contents of your
`~/.kaggle/access_token`) and tick it for this notebook.
To stop the queue: stop this notebook's run on its Kaggle page.

In [ ]:
RUN_PROFILE = "full"   # "full" on GPU; "preflight" = quick CPU smoke test (tools/push.py)
PREFLIGHT = RUN_PROFILE == "preflight"
import os, sys, glob, json, time, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def _find_lib():
    for c in ("/kaggle/input/asd-xmf-lib", "/kaggle/input/datasets/shazedaoni/asd-xmf-lib"):
        if os.path.exists(os.path.join(c, "asd_utils.py")):
            return c
    for depth in range(1, 6):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + "asd_utils.py")
        if hits:
            return os.path.dirname(hits[0])
    return os.path.abspath(os.path.join(os.getcwd(), "..", "..", "asd_lib"))  # local run

def _require_internet():
    import socket
    try:
        socket.create_connection(("pypi.org", 443), timeout=10).close()
    except OSError:
        raise SystemExit("NO INTERNET in this Kaggle session: verify the account's phone number (Kaggle > Settings) "
                         "and keep 'Internet on'. Stopping so no step runs with missing models or features.")
_require_internet()

LIB = _find_lib()
sys.path.insert(0, LIB)
import asd_utils as U
print("asd_lib:", LIB, "| profile:", RUN_PROFILE)
ENV = U.env_report()
open(os.path.join(str(U.WORK), "_run_profile_" + RUN_PROFILE + ".txt"), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))  # queue marker

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
except Exception as e:
    TOKEN = None
    print("Secret KAGGLE_API_TOKEN is not attached to this notebook:", repr(e)[:200])
    print("Add it under Add-ons -> Secrets, tick it for this notebook, then Save & Run All once.")

In [ ]:
if TOKEN:
    os.environ["KAGGLE_API_TOKEN"] = TOKEN.strip()
    kd = Path.home() / ".kaggle"; kd.mkdir(exist_ok=True)
    (kd / "access_token").write_text(TOKEN.strip()); os.chmod(kd / "access_token", 0o600)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=False)
    bundle = U.find_path("queue.json").parent.parent          # <bundle>/tools/queue.json
    os.environ.update({"ASD_QUEUE_ROOT": str(bundle), "ASD_QUEUE_FILE": str(bundle / "tools" / "queue.json"),
                       "ASD_QUEUE_LOG": "/kaggle/working/queue_log.txt",
                       "ASD_SELF_DIR": str(bundle / "kaggle_build" / "asd-xmf-queue-orchestrator")})
    sys.path.insert(0, str(bundle / "tools"))
    import queue_runner as Q
    print("kaggle CLI:", subprocess.run(["kaggle", "--version"], capture_output=True, text=True).stdout.strip())
    Q.loop(max_hours=0.2 if PREFLIGHT else 11.0, self_slug="asd-xmf-queue-orchestrator")